# TP5 · Ejercicio 4 — banco de pruebas

El Ejercicio 4 es **teórico**: se responde con teoría de Constraint Programming. Este notebook
no trae la respuesta; te da **evidencia** para apoyarla y un lugar donde escribirla.

**Antes de correrlo**, las Tareas 1 y 2 tienen que estar en verde
(`uv run pytest tests/test_tsp.py`): el experimento usa tus funciones de `models/tsp.py`.

Para abrirlo, desde `labs/cp`:

```bash
uv run --group notebook jupyter lab notebooks/tp5-ejercicio4.ipynb
```

## Qué leer (todo en `materials/`)

| Para | Dónde |
|---|---|
| Si CP te suena a chino: variables, dominios, búsqueda y propagación | `week-03/CS-Topics-Week3-1.pdf`, pp. 7–35 |
| Propagación y consistencia de arco (*arc consistency*) | `week-04/CS-Topics-Week4-1.pdf`, pp. 20–58 |
| Subtours, restricciones de eliminación (DFJ), *branch-and-cut* | `week-05/Topicos_CC-5.pdf`, pp. 13–31 y 45 |
| MTZ (*timing constraints*) y cómo pasar de un «si…» a una desigualdad | `week-05/Topicos_CC-5.pdf`, pp. 32–44 y 48–49 |

In [1]:
import math
import random
import sys
from pathlib import Path

# labs/cp, para poder importar models/ sin importar desde dónde arrancó el kernel
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models" / "tsp.py").exists())
sys.path.insert(0, str(RAIZ))

from ortools.sat.python import cp_model

from models.tsp import extraer_ruta, solve_tsp_circuit_basic, solve_tsp_mtz_cp

## Herramientas de medición

`medir` corre un modelo con un límite de tiempo y devuelve lo que reporta CP-SAT:

| Columna | Qué es |
|---|---|
| `estado` | `OPTIMAL`: encontró una ruta **y demostró** que no hay otra mejor. `FEASIBLE`: encontró una ruta, pero se acabó el tiempo antes de demostrarlo. `UNKNOWN`: no encontró ninguna. |
| `costo` | costo de la mejor ruta encontrada |
| `cota` | mejor cota inferior demostrada; si `costo == cota`, esa ruta es la óptima |
| `tiempo_s` | segundos de reloj |
| `ramas` | decisiones que tomó la búsqueda |
| `conflictos` | veces que la búsqueda llegó a un callejón sin salida y retrocedió |
| `ruta_ok` | la ruta pasa por todas las ciudades una sola vez (sin subtours) |

In [2]:
def instancia(n: int, seed: int = 0) -> list[list[int]]:
    """n ciudades al azar en un cuadrado de 1000 x 1000; distancia euclidiana redondeada."""
    rng = random.Random(seed)
    puntos = [(rng.uniform(0, 1000), rng.uniform(0, 1000)) for _ in range(n)]
    return [[round(math.dist(p, q)) for q in puntos] for p in puntos]


def medir(solve, dist: list[list[int]], limite_s: float = 20.0) -> dict:
    """Corre `solve` con un límite de tiempo y devuelve lo que reporta el solver."""
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = limite_s
    resultado = solve(dist, solver=solver)
    n = len(dist)
    return {
        # status_name() sin argumento falla en OR-Tools 9.15; se le pasa el estado explícito
        "estado": solver.status_name(solver.response_proto.status),
        "costo": resultado[0] if resultado else None,
        "cota": round(solver.best_objective_bound),
        "tiempo_s": round(solver.wall_time, 2),
        "ramas": solver.num_branches,
        "conflictos": solver.num_conflicts,
        "ruta_ok": resultado is not None and sorted(resultado[1][:-1]) == list(range(n)),
    }


def tabla(filas: list[dict]) -> None:
    cols = list(filas[0])
    ancho = {c: max(len(c), *(len(str(f[c])) for f in filas)) for c in cols}
    print("  ".join(c.ljust(ancho[c]) for c in cols))
    for f in filas:
        print("  ".join(str(f[c]).ljust(ancho[c]) for c in cols))

## 4.1 — Eficiencia: `AddCircuit` vs MTZ con n = 50

Los dos modelos, la misma instancia y el mismo límite de tiempo, subiendo n hasta el 50 del
enunciado. En el peor caso la celda tarda `LIMITE_S` × 8 segundos.

In [3]:
LIMITE_S = 20
MODELOS = {"AddCircuit": solve_tsp_circuit_basic, "MTZ reificado": solve_tsp_mtz_cp}

filas = []
for n in [10, 20, 30, 50]:
    dist = instancia(n)
    for nombre, solve in MODELOS.items():
        filas.append({"n": n, "modelo": nombre, **medir(solve, dist, LIMITE_S)})
tabla(filas)

NotImplementedError: Tarea 1, paso 4: extraer la ruta

**Qué mirar en la tabla**

- ¿Qué modelo llega a `OPTIMAL` para cada n, y en cuánto tiempo?
- Cuando uno no llega, ¿qué tan lejos queda su `costo` de su `cota`?
- ¿Cómo crecen `ramas` y `conflictos` con n en cada modelo?

La tabla dice **qué** pasa. El **porqué** (cómo poda los dominios cada modelo) es lo que pide el
enunciado, y sale de las lecturas de arriba. La documentación de la restricción, por si ayuda:

In [ ]:
help(cp_model.CpModel.add_circuit)

### Respuesta 4.1

_Escribe aquí tu respuesta._

## 4.2 — Reificación vs Big-M

En MIP, MTZ se escribe con una constante grande: $u_i - u_j + n \cdot x_{ij} \le n - 1$.
La pregunta es teórica: por qué en CP-SAT es más limpio usar `only_enforce_if` que esa constante.

**Opcional**, si quieres ver las dos versiones lado a lado: completa esta variante de la Tarea 2
con la desigualdad Big-M del enunciado y compárala con la tuya. Mientras no la completes, el
modelo acepta subtours y lo verás como `ruta_ok = False`.

In [ ]:
def solve_tsp_mtz_bigm(dist_matrix: list[list[int]], solver: cp_model.CpSolver | None = None):
    """Tarea 2 con la desigualdad Big-M del enunciado en vez de only_enforce_if."""
    n = len(dist_matrix)
    model = cp_model.CpModel()
    x = {(i, j): model.new_bool_var(f"x_{i}_{j}") for i in range(n) for j in range(n) if i != j}
    u = {i: model.new_int_var(1, n - 1, f"u_{i}") for i in range(1, n)}

    for i in range(n):
        model.add_exactly_one(x[i, j] for j in range(n) if i != j)
        model.add_exactly_one(x[j, i] for j in range(n) if i != j)

    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                # COMPLETAR AQUI: la desigualdad Big-M del enunciado, sin only_enforce_if
                pass

    model.minimize(sum(x[i, j] * dist_matrix[i][j] for i, j in x))

    solver = solver if solver is not None else cp_model.CpSolver()
    if solver.solve(model) not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None
    return int(solver.objective_value), extraer_ruta(solver, x, n)

In [ ]:
filas = []
for n in [8, 12, 16, 20]:
    dist = instancia(n)
    for nombre, solve in {"only_enforce_if": solve_tsp_mtz_cp, "Big-M": solve_tsp_mtz_bigm}.items():
        filas.append({"n": n, "modelo": nombre, **medir(solve, dist, LIMITE_S)})
tabla(filas)

Puede que en instancias chicas el cronómetro no muestre mucha diferencia: la pregunta pide el
argumento teórico, y la tabla es solo apoyo.

### Respuesta 4.2

_Escribe aquí tu respuesta._